In [14]:
%run ../module7-genai-langchain/initialize_environment.py


MCP Windows stderr patch applied.
Environment initializing completed successfully.


# Multi-Agent Wedding Planner

In [13]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [12]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [11]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [15]:
# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [16]:
# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [17]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [18]:
from langchain.agents import create_agent

coordinator = create_agent(
    model=create_azure_llm(),
    tools=[search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


## Test


In [43]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from Lahore and I'd like a wedding in Gujranwala for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)
print(response["messages"][-1].content)

For your wedding in Gujranwala with 100 guests and a jazz music genre, I found the following options:

Venues:
1. Adam Palace Marquee - AC hall, beautiful decoration, food service, affordable.
2. Lone Palace Marriage Hall - Elegant, modern amenities, mid-range price.
3. Meridian Hotel Gujranwala - Luxurious hotel venue, banquet and marquee available, approx Rs. 2375 per head.
4. Golf Club Marquee Rahwali Cantt - Premium venue, open-air event space, Rs. 1150 per head plus extra charges.

Playlist:
I curated a jazz playlist with 10 tracks totaling about 35 minutes and 36 seconds, costing $9.90. 

Would you like to proceed with booking any of these venues? And would you like to add more songs or keep the playlist as is?


In [44]:
response = await coordinator.ainvoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "I'm from Lahore and I'd like a wedding in Gujranwala for 100 guests, jazz-genre."
                )
            )
        ],
    },
    config={"tags": ["WP"], "recursion_limit": 40},
)

print(response["messages"][-1].content)

For your wedding in Gujranwala with 100 guests and a jazz music theme, here are the venue options:

1. Adam Palace Gujranwala - air-conditioned hall, beautiful decorations, food service included, highly regarded.
2. Lone Palace Marriage Hall - elegant venue with modern amenities, personalized services.
3. Meridian Hotel Gujranwala - luxurious with banquet and marquee options, approx. Rs. 2375 per head.
4. Golf Club Marquee, Rahwali Cantt - elegant open-air and indoor options, Rs. 1150 per head plus kitchen charges, highly praised.

For the playlist, I have found a great selection of jazz tracks with details available.

Would you like to proceed with booking one of these venues? If so, which one? Also, do you want me to finalize the jazz playlist for your wedding?


In [45]:
response = await coordinator.ainvoke(
    {
        "messages": response["messages"] + [
            HumanMessage(
                content="Please share the playlist (song names, durations, prices, total duration, and total cost)."
            )
        ]
    },
    config={"tags": ["WP"], "recursion_limit": 40},
)

print(response["messages"][-1].content)

Here is the jazz playlist curated for your wedding along with song names, durations, and prices:

1. Desafinado - 185,338 ms (~3 min 5 sec) - $0.99
2. Garota De Ipanema - 285,048 ms (~4 min 45 sec) - $0.99
3. Samba De Uma Nota Só (One Note Samba) - 137,273 ms (~2 min 17 sec) - $0.99
4. Por Causa De Você - 169,900 ms (~2 min 50 sec) - $0.99
5. Ligia - 251,977 ms (~4 min 12 sec) - $0.99
6. Fotografia - 129,227 ms (~2 min 9 sec) - $0.99
7. Dindi (Dindi) - 253,178 ms (~4 min 13 sec) - $0.99
8. Se Todos Fossem Iguais A Você (Instrumental) - 134,948 ms (~2 min 15 sec) - $0.99
9. Falando De Amor - 219,663 ms (~3 min 40 sec) - $0.99
10. Angela - 169,508 ms (~2 min 50 sec) - $0.99

Total duration: approximately 21 minutes and 16 seconds
Total cost: $9.90

Would you like me to assist with creating the final playlist for your wedding or proceed with booking a venue?


In [46]:
response = await coordinator.ainvoke(
    {
        "message": response["messages"].append(
             HumanMessage(
                content=("I want to book the first avenue")
            )
        )
    },
    config={"tags": ["WP"], "recursion_limit": 40},
)

print(response["messages"][-1].content)

To start coordinating your perfect wedding, I need some details:

1. What is the origin location for travel arrangements?
2. What is the destination for the wedding?
3. How many guests will be attending?
4. What music genre do you prefer for the wedding playlist?

Please provide this information so I can update the state and proceed.
